# Compare classical image classifiers

Classify handwritten digits using fixed pixel measurements and classical ML. Reserve a final holdout, select a tree ensemble using development folds, compare a linear baseline on the identical holdout, and explain failure cases. This project uses no neural networks.

Dataset: scikit-learn digits benchmark · 1,797 observations, 64 pixel measurements

Reference: https://scikit-learn.org/stable/datasets/toy_dataset.html#optical-recognition-of-handwritten-digits-dataset

Original ML Atlas notebook, MIT-licensed code and CC BY 4.0 explanation. Third-party data retains its own license.


## Environment

Run in Jupyter or Colab. For the scikit-learn projects, install scikit-learn >=1.4 and its dependencies in your own environment. No GPU is needed.


## Plan

1. Inspect feature ranges, target counts and the fixed 8×8 representation.
2. Reserve a stratified 20% final holdout before comparing procedures.
3. Tune minimum leaf size for a random forest using stratified training folds; fit a scaled logistic baseline separately.
4. Compare held-out accuracy, macro F1 and confusion counts on identical observations.
5. Inspect misclassified examples and report the limits of random-split performance on this historical benchmark.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=.2, stratify=data.target, random_state=42)
folds = StratifiedKFold(3, shuffle=True, random_state=42)
search = GridSearchCV(RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=1), {'min_samples_leaf': [1, 3, 5]}, cv=folds, scoring='f1_macro', n_jobs=1)
search.fit(X_train, y_train)
linear = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
linear.fit(X_train, y_train)
print('Train/test:', X_train.shape, X_test.shape)
print('Selected forest:', search.best_params_)
for name, model in [('Linear', linear), ('Forest', search.best_estimator_)]:
    predicted = model.predict(X_test)
    print(name, 'accuracy:', round(accuracy_score(y_test, predicted), 4), 'macro F1:', round(f1_score(y_test, predicted, average='macro'), 4))
    print(confusion_matrix(y_test, predicted))
    print('First errors (actual, predicted):', [(int(a), int(b)) for a,b in zip(y_test,predicted) if a != b][:5])


## Review the result

- [ ] The final holdout never guides tree tuning.
- [ ] Both models use the same evaluation records.
- [ ] The report includes a linear baseline and macro F1, not only accuracy.
- [ ] At least three errors are examined with a proposed explanation.
- [ ] The environment and random seeds are documented.


## Extend it

Compare a scaled k-nearest-neighbor pipeline using the same development folds. Measure prediction latency and investigate whether preprocessing changes the result.

Record what changed, why, and how you evaluated it.
